# The closed test set — `test.ipynb`

The one number this direction owes and that `train.ipynb` cannot produce: the model on
`subject_test.json`. [`docs/evaluation.md`](../../../docs/evaluation.md) keeps that file for the
end — 27 subjects, scored **once**, with the configuration chosen before it is opened. This
notebook is that single opening for `tia`.

It retrains the same five subject folds the board quotes (same seeds, same recipe), scores the
closed set with each fitted model at the threshold that model chose on its own `val`, and writes
`results/<variant>-test/`.

Three rules keep the number honest, and all three are in the code:

1. **Nothing about the test set chooses anything.** The threshold comes from `val`, the epoch
   from `val`, the variant from the cell above. The test set is scored, never used.
2. **The folds are retrained, not reloaded** (~10 s a fold). The holdout column of those same
   folds is printed beside the test column, so the retraining can be checked against the run the
   board quotes.
3. **One configuration.** If the number disappoints, that is the result. Re-running this notebook
   with another `VARIANT` is tuning on the test set, and it destroys the estimate: the ±18.5 pp
   interval of `evaluation.md` is what 27 subjects buy you, and picking the best of several draws
   is not inside it.

## 0 — Setup

In [1]:
# Colab: mount Drive and keep the repo current. Locally both are skipped: the cells below find
# the checkout and the shared folder on their own. No magic commands, so this runs anywhere.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    COLAB = True
except ImportError:
    COLAB = False

import os
import subprocess
import sys
from pathlib import Path

REPO = "/content/DeepLearningCourse"
if COLAB:
    if not os.path.isdir(f"{REPO}/.git"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/mattsays/DeepLearningCourse", REPO], check=True)
    subprocess.run(["git", "pull", "-q"], cwd=f"{REPO}/lie-detector", check=False)
    os.chdir(f"{REPO}/lie-detector")

# The `lib/` package lives in `lie-detector/`: walk up until the folder is found, so the
# notebook runs from any kernel starting point.
HERE = Path.cwd()
while not (HERE / "lib" / "dataset.py").exists() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE))
print("colab:", COLAB, "| lie-detector:", HERE)

Mounted at /content/drive
colab: True | lie-detector: /content/DeepLearningCourse/lie-detector


## 1 — The configuration, frozen before the set is opened

`VARIANT` is the model whose fold run the board quotes (`results/late-subject`), and `SEEDS` are
the seeds of that run: the test set is scored by the *same* models, otherwise the two numbers
describe two different things. Changing either here means choosing a new configuration, which is
allowed **only before** the guard cell below is unlocked.

In [ ]:
from pathlib import Path

# The one shared folder (docs/data.md). On Colab it is a Drive shortcut named "dolos".
SHARED = Path("/content/drive/MyDrive/dolos")
DATA = SHARED if SHARED.exists() else Path("data/processed")
print("data:", DATA, "->", sorted(p.name for p in DATA.glob("*"))[:8] if DATA.exists() else "missing")

DIRECTION = "tia"
VARIANT = "video"                   # the row being tested: results/late-subject
PROTOCOL = "v1"                    # "v1" board protocol; "v2" is the purity-balanced ablation
CAPTAINS = "include"               # as shipped; "exclude" tests the nocaptains ablation
SEEDS = [0, 1, 2]                  # the seeds of the run this number is compared with
ENCODERS = {"video": "dinov2-small", "audio": "wavlm-base-plus"}

BATCH_SIZE, EPOCHS, PATIENCE = 16, 30, 8     # the recipe of the board runs, unchanged
READ_WORKERS = 64                            # parallel readers for the latency-bound Drive mount

data: /content/drive/MyDrive/dolos -> ['audio', 'cached_features', 'clips', 'dataset.csv', 'dolos_annotations.zip', 'splits', 'splits_v2']


In [3]:
NOTEBOOK_DIR = HERE / "notebooks/architectures/cross-modal-fusion"
sys.path.insert(0, str(NOTEBOOK_DIR))

import model
from training import Data, Encoders, Settings

settings = Settings(data=DATA, encoders=Encoders(**ENCODERS), seeds=list(SEEDS),
                    epochs=EPOCHS, patience=PATIENCE, batch_size=BATCH_SIZE,
                    read_workers=READ_WORKERS, protocol=PROTOCOL, captains=CAPTAINS)
print("notebook dir:", NOTEBOOK_DIR)
print("device:", settings.torch_device())

notebook dir: /content/DeepLearningCourse/lie-detector/notebooks/architectures/cross-modal-fusion
device: cuda


## 2 — The protocol and the closed set

`Protocol.load` verifies the manifest hash, so a hand-edited split raises instead of silently
scoring a different protocol. `subject_test.json` is read here for the first time — and the three
assertions below are what make the number mean something:

- **no test subject is in any fold**, so the retraining never sees them;
- **no test clip is in any fold**, the clip-level version of the same check;
- **no captain is in the test set**: they are pinned to the training side by construction, and
  the split script already refuses to write a test set that contains one.

The 47.5% deception of `docs/splits.md` is matched in the working pool on purpose, so the majority
baseline of this set is not a free win.

That file measures this protocol at **1,021 working clips plus the two captains, and 182 test
clips in 27 subjects**. If the counts printed below are smaller, or the two fingerprints differ
from `results/<variant>-subject/run.json`, then `DATA` is not the folder that produced the board
row: stop and mount the right one. A test number computed on another population is not comparable
with the board, and the set is spent either way.

In [4]:
from lib.dataset import Clips, Protocol

folds = Protocol.load(settings.splits())
clips = Clips.load(DATA)
print(clips.describe_source())          # post it: same fingerprint = same dataset
print("split dir:", settings.splits(), "| manifest verified:", folds.verified)

test_ids = clips.keep(folds.test())
test_subjects = {clips[c].subject for c in test_ids}
fold_ids, fold_subjects = set(), set()
for fold in range(1, folds.folds + 1):
    for part in folds.subject(fold):
        fold_ids |= set(part)
        fold_subjects |= {clips[c].subject for c in part}

assert not (test_subjects & fold_subjects), \
    f"test subjects inside a fold: {sorted(test_subjects & fold_subjects)[:5]}"
assert not (set(test_ids) & fold_ids), "a test clip sits inside a fold"
assert not (test_subjects & set(folds._splits["subject_fold1.json"]["always_train"]["subjects"])), \
    "a captain is in the test set"

split_test, split_subjects = folds.test(), folds._splits["subject_test.json"]["subjects"]
stats = clips.describe(test_ids)
print(f"\ntest set: {len(test_ids)}/{len(split_test)} clips of "
      f"{len(test_subjects)}/{len(split_subjects)} subjects, "
      f"{stats['deception']}% deception, {stats['female']}% female, "
      f"majority baseline {stats['majority_baseline']}%")
print(f"{len(fold_subjects)} working subjects in the folds, no overlap with the test")

# Provenance. `export.describe` is the pair of fingerprints every run.json carries: a split
# folder rebuilt from a different dataset.csv keeps the same file names and a different
# population, so the names identify nothing — the hashes do.
import json

from export import describe, run_name

now = describe(clips, folds)
print(f"\ndataset {now['dataset']['fingerprint']}  "
      f"splits {now['protocol_files']['manifest_sha256']}")

dropped = sorted((set(fold_ids) | set(split_test)) - set(clips.ids()))
if dropped:
    print(f"⚠️  {len(dropped)} clips of the frozen splits are absent from dataset.csv and would "
          f"be dropped silently: the folds and the test set shrink with them.")

stored_run = (NOTEBOOK_DIR / "results" / run_name(VARIANT, "subject", PROTOCOL, CAPTAINS)
              / "run.json")
if stored_run.exists():
    stored = json.loads(stored_run.read_text(encoding="utf-8"))
    same_data = stored["dataset"]["fingerprint"] == now["dataset"]["fingerprint"]
    same_splits = (stored["protocol_files"].get("manifest_sha256")
                   == now["protocol_files"]["manifest_sha256"])
    if not (same_data and same_splits):
        print(f"⚠️  this is not the DATA behind {stored_run.parent.name}: "
              f"dataset {'same' if same_data else 'DIFFERENT'}, "
              f"splits {'same' if same_splits else 'DIFFERENT'}.\n"
              "    The folds retrained below and the closed set would be a different population, "
              "so the test number would not be comparable with the board row — and not the set "
              "docs/splits.md describes. Point DATA at the shared folder that produced run.json "
              "before opening the set.")

dataset.csv sha256:8fa198b1fe71  1678 clips  1452 with media
split dir: /content/drive/MyDrive/dolos/splits | manifest verified: True

test set: 155/155 clips of 24/24 subjects, 47.7% deception, 28.4% female, majority baseline 52.3%
137 working subjects in the folds, no overlap with the test

dataset 8fa198b1fe71  splits fbc88b132eb7


## 3 — The guard

Everything above this cell is still free to change. Unlocking the cell is the decision
`evaluation.md` asks for: **the configuration is final, and the numbers that come out are the
ones that get reported**. Set `OPEN_TEST_SET = True` and run the notebook top to bottom in one
sitting — a partial run leaves the set opened without a number.

In [5]:
print(f"about to test {VARIANT} ({model.VARIANTS[VARIANT].__name__}) on {PROTOCOL}/{CAPTAINS}")
print(f"  {len(SEEDS)} seed(s) x {folds.folds} folds trained on {len(fold_ids)} working clips")
print(f"  then scored on {len(test_ids)} test clips of {len(test_subjects)} subjects")
print(f"  encoders: {ENCODERS}")

# docs/evaluation.md: opened once, with the configuration chosen before. After this cell the
# number exists; changing anything afterwards is tuning on the test set, not the end of the run.
OPEN_TEST_SET = True

if not OPEN_TEST_SET:
    raise SystemExit("the test set is opened once: set OPEN_TEST_SET = True in this cell when "
                     "the configuration above is final, then run the notebook top to bottom")

about to test late (LateFusion) on v1/include
  3 seed(s) x 5 folds trained on 1297 working clips
  then scored on 155 test clips of 24 subjects
  encoders: {'video': 'dinov2-small', 'audio': 'wavlm-base-plus'}


## 4 — The feature store

One `Data` for the whole notebook: the two caches are read once, and the split is `"test"` —
the working clips *plus* the closed set, because the folds train on the first and are scored on
the second. `Data` prints which caches it read; a test clip with no cached array is dropped, and
the count below says by how many. If it is not zero, the number is on a partial test set and the
report has to say so.

In [6]:
data = Data(clips, folds, settings, split="test")
usable = data.features.subset(test_ids)
print(f"test clips with all three cached arrays: {len(usable)}/{len(test_ids)}")
if len(usable) < len(test_ids):
    absent = sorted(set(test_ids) - set(usable))
    print(f"⚠️  {len(absent)} test clips have no cached features: the number is on a PARTIAL "
          f"test set — say so next to it. First: {absent[:3]}")

[test] 1452 clips in the protocol, reading both caches once
  video 1299 read, 153 missing | audio 1299 read, 153 missing
  usable: 1299/1452 clips
test clips with all three cached arrays: 141/155
⚠️  14 test clips have no cached features: the number is on a PARTIAL test set — say so next to it. First: ['an_wilty_ep18_truth1', 'an_wilty_ep18_truth2', 'an_wilty_ep18_truth3']


## 5 — Train the folds, score the closed set

One call: every seed × fold is trained with the board recipe (early stopping on validation AUC,
`pos_weight` for the class prior, threshold chosen on `val`), then scored on the test set at that
fold's own validation threshold. The holdout of the same model is returned beside it — that is
the column §6 compares with the stored run, and the board's own number.

The test set has no early stopping, no threshold search and no model selection in this loop. That
is the whole point of it.

In [7]:
from training import run_test

test_summary, test_preds, holdout_summary = run_test(data, VARIANT)

    fold 1 seed 0: ACC  60.5  AUC  64.9  thr 0.37    15s
    fold 1 seed 0 on the test set: ACC  43.3  AUC  43.0  thr 0.37
    fold 2 seed 0: ACC  52.9  AUC  57.9  thr 0.41     8s
    fold 2 seed 0 on the test set: ACC  44.0  AUC  49.4  thr 0.41
    fold 3 seed 0: ACC  59.8  AUC  45.3  thr 0.53     4s
    fold 3 seed 0 on the test set: ACC  49.6  AUC  47.9  thr 0.53
    fold 4 seed 0: ACC  51.5  AUC  53.7  thr 0.21     8s
    fold 4 seed 0 on the test set: ACC  51.1  AUC  52.4  thr 0.21
    fold 5 seed 0: ACC  57.9  AUC  65.0  thr 0.84    10s
    fold 5 seed 0 on the test set: ACC  51.8  AUC  54.2  thr 0.84
    fold 1 seed 1: ACC  67.3  AUC  69.6  thr 0.46    12s
    fold 1 seed 1 on the test set: ACC  47.5  AUC  48.4  thr 0.46
    fold 2 seed 1: ACC  56.7  AUC  57.2  thr 0.63    12s
    fold 2 seed 1 on the test set: ACC  44.7  AUC  45.4  thr 0.63
    fold 3 seed 1: ACC  48.2  AUC  43.1  thr 0.44     3s
    fold 3 seed 1 on the test set: ACC  46.1  AUC  48.0  thr 0.44
    fold 4 seed 

## 6 — Is this the model the board quotes?

The folds are retrained from scratch, so the test number belongs to a run that must be *checked*
to be the board's. `results/<variant>-subject/folds.csv` holds the holdouts of that run; if the
same seeds, encoders and split reproduce them, the two numbers describe the same model.

They should agree to the digit on the same GPU and the same code. A gap of a point or two is
cuDNN non-determinism or a moved git checkout; a gap of five is a different configuration, and
the warning below says what to look at. This is a check, not a metric: it is never quoted.

In [8]:
import pandas as pd

from export import run_name

reference_dir = NOTEBOOK_DIR / "results" / run_name(VARIANT, "subject", PROTOCOL, CAPTAINS)
reference = reference_dir / "folds.csv"
if reference.exists():
    stored = pd.read_csv(reference)
    both = holdout_summary.merge(stored, on=["seed", "fold"], suffixes=("", "_stored"))
    acc_gap = float((both.acc - both.acc_stored).abs().max())
    auc_gap = float((both.auc - both.auc_stored).abs().max())
    print(f"retrained holdouts vs {reference_dir.name}/folds.csv on {len(both)} folds: "
          f"max |ΔACC| {acc_gap * 100:.1f} pp, max |ΔAUC| {auc_gap * 100:.1f} pp")
    if max(acc_gap, auc_gap) > 0.02:
        print("⚠️  the folds did not reproduce the stored run: this is not the board model's "
              "test number. Check ENCODERS, PROTOCOL and the git_commit in "
              f"{reference_dir.name}/run.json before quoting it.")
else:
    print(f"no {reference} — nothing to compare with; the test number still stands on its own")

retrained holdouts vs late-subject/folds.csv on 15 folds: max |ΔACC| 0.0 pp, max |ΔAUC| 0.0 pp


## 7 — Performance on the test set

One row per trained model. The spread is the model's, not the estimate's: it says how much the
number moves with the seed and the fold, and it is not the uncertainty of the test score.

In [9]:
columns = ["seed", "fold", "acc", "f1", "auc", "balanced_acc", "threshold"]
print(test_summary[columns].round(3).to_string(index=False))
print()
print(f"test ACC {test_summary.acc.mean() * 100:5.1f} ± {test_summary.acc.std() * 100:.1f}  "
      f"F1 {test_summary.f1.mean() * 100:5.1f}  AUC {test_summary.auc.mean() * 100:5.1f}  "
      f"balanced {test_summary.balanced_acc.mean() * 100:5.1f}")
print(f"({len(test_summary)} models: {len(SEEDS)} seed(s) x {folds.folds} folds)")

 seed  fold   acc    f1   auc  balanced_acc  threshold
    0     1 0.433 0.429 0.430         0.435      0.372
    0     2 0.440 0.564 0.494         0.465      0.406
    0     3 0.496 0.283 0.479         0.476      0.526
    0     4 0.511 0.561 0.524         0.523      0.211
    0     5 0.518 0.320 0.542         0.498      0.839
    1     1 0.475 0.464 0.484         0.476      0.465
    1     2 0.447 0.458 0.454         0.451      0.627
    1     3 0.461 0.321 0.480         0.448      0.443
    1     4 0.482 0.459 0.478         0.482      0.338
    1     5 0.511 0.582 0.492         0.527      0.164
    2     1 0.454 0.430 0.494         0.453      0.475
    2     2 0.546 0.590 0.532         0.558      0.617
    2     3 0.454 0.533 0.432         0.470      0.066
    2     4 0.546 0.628 0.615         0.567      0.062
    2     5 0.511 0.606 0.491         0.533      0.155

test ACC  48.6 ± 3.7  F1  48.2  AUC  49.5  balanced  49.1
(15 models: 3 seed(s) x 5 folds)


## 8 — One prediction per clip, and how uncertain it is

The models are on the same 27 subjects but saw different training folds, so their errors overlap;
averaging them is the closest thing to a single decision this protocol offers. Two versions, to
show the number is not an artefact of the threshold: the mean probability at the mean validation
threshold, and the majority vote of the models at their own thresholds.

The interval is a **cluster bootstrap over subjects**, not over clips: clips of one speaker are
not independent, and `lib/metrics.cluster_bootstrap` is the team's answer to that. Twenty-seven
resampled subjects is a small sample — the interval is wide, and that width is the result.

In [ ]:
import pandas as pd

from lib.metrics import cluster_bootstrap, evaluate, majority_baseline, subject_level

ensemble = (test_preds.groupby("clip_id", as_index=False)
                      .agg(y_true=("y_true", "first"), y_prob=("y_prob", "mean")))
threshold = float(test_summary.threshold.mean())

meta = pd.read_csv(DATA / "dataset.csv", dtype=str).set_index("clip_id")
subjects = ensemble.clip_id.map(meta.subject)

pooled = evaluate(ensemble.y_true, ensemble.y_prob, threshold=threshold)
ci = cluster_bootstrap(ensemble.y_true, ensemble.y_prob, subjects, n_boot=2000, seed=0,
                       threshold=threshold)
print(f"ensemble of {len(test_summary)} models at threshold {threshold:.2f} "
      f"(mean of the per-model validation thresholds)")
for metric in ("acc", "f1", "auc", "balanced_acc"):
    print(f"  {metric:13} {pooled[metric] * 100:5.1f}   95% CI "
          f"[{ci[metric]['lo'] * 100:5.1f}, {ci[metric]['hi'] * 100:5.1f}]")

votes = test_preds.merge(test_summary[["seed", "fold", "threshold"]], on=["seed", "fold"])
votes["vote"] = (votes.y_prob >= votes.threshold).astype(int)
voted = votes.groupby("clip_id", as_index=False).agg(y_true=("y_true", "first"),
                                                     vote=("vote", "mean"))
voted_metrics = evaluate(voted.y_true, voted.vote, threshold=0.5)
print(f"majority vote of the same models: ACC {voted_metrics['acc'] * 100:5.1f}  "
      f"AUC {voted_metrics['auc'] * 100:5.1f}")

per_subject = subject_level(ensemble.y_true, ensemble.y_prob, subjects, threshold=threshold)
print(f"\n{per_subject['n_subjects']} subjects: mean per-subject ACC "
      f"{per_subject['subject_acc'] * 100:.1f}, one vote per subject "
      f"{per_subject['subject_vote_acc'] * 100:.1f}, pooled clips "
      f"{per_subject['clip_acc'] * 100:.1f}")
print(f"majority baseline on these clips: {majority_baseline(ensemble.y_true) * 100:.1f}")

## 9 — Who it gets wrong

A pooled ACC can hide a model that is balanced on paper and useless in practice: male clips are
79% of the dataset and 56.6% deception against 43.8% for female. The per-gender split and the
confusion matrix are what the report's error section reads.

In [ ]:
confusion = pooled["confusion"]
print(f"confusion (ensemble): tn {confusion['tn']}  fp {confusion['fp']}  "
      f"fn {confusion['fn']}  tp {confusion['tp']}")

ensemble["gender"] = ensemble.clip_id.map(meta.gender)
for gender, group in ensemble.groupby("gender"):
    m = evaluate(group.y_true, group.y_prob, threshold=threshold)
    print(f"{gender:7} n={m['n']:3d}  ACC {m['acc'] * 100:5.1f}  "
          f"balanced {m['balanced_acc'] * 100:5.1f}  AUC {m['auc'] * 100:5.1f}")

## 10 — The test number beside the board number

Read the first two rows as the same quantity measured on different people; read the third as a
different quantity (pooled over clips, not a mean over folds) that is more precise but not
comparable with them.

`evaluation.md` gives the interval the board's 151 subjects buy: ±7.8 pp. The test set has 27, so
its interval is ±18.5 pp — the two estimates overlapping is expected, and a gap of a few points
between them is not evidence of anything.

In [ ]:
import json

stored_summary = reference_dir / "summary.json"
if stored_summary.exists():
    board = json.loads(stored_summary.read_text(encoding="utf-8"))
    rows = [
        {"estimate": f"board: {folds.folds} holdouts x {len(SEEDS)} seeds",
         "unit": "mean of per-fold", "acc": board["acc"], "f1": board["f1"],
         "auc": board["auc"], "balanced_acc": board["balanced_acc"]},
        {"estimate": "test: the same folds", "unit": "mean of per-model",
         "acc": test_summary.acc.mean(), "f1": test_summary.f1.mean(),
         "auc": test_summary.auc.mean(), "balanced_acc": test_summary.balanced_acc.mean()},
        {"estimate": "test: ensembled", "unit": "pooled over clips",
         "acc": pooled["acc"], "f1": pooled["f1"], "auc": pooled["auc"],
         "balanced_acc": pooled["balanced_acc"]},
    ]
    print(pd.DataFrame(rows).round(3).to_string(index=False))
    print(f"\n{board['n_folds']} folds, {board['n_seeds']} seeds on the board side; "
          f"{len(ensemble)} test clips here")
else:
    print(f"no {stored_summary}: run train.ipynb or campaign.ipynb for this configuration first")

## 11 — Save the run

Written through `export.save_run` like every other run, so `results/README.md` stays the only
description of the layout. The one difference: here `preds.csv` has **one row per model and
clip**, because every fold scores every test clip — the ensemble used above is its
`groupby("clip_id").y_prob.mean()`. `board_row.md` says what the directory is, so nobody pastes
it on the board.

In [ ]:
from export import save_run
from training import config_for

config = config_for(data, VARIANT, settings, direction=DIRECTION, owner="mattsays")
directory = save_run(test_summary, test_preds, config, root=NOTEBOOK_DIR / "results")
print(f"{len(test_preds)} prediction rows over {ensemble.clip_id.nunique()} test clips")

## How to read the number

**The headline is the mean over models of ACC / F1 / AUC / balanced ACC on the closed set, with
the cluster-bootstrap interval from §8.** Quote it with three things beside it: the number of
test clips and subjects it was computed on, the `dataset.fingerprint` and
`protocol_files.manifest_sha256` in `run.json`, and the fact that it is one configuration.

**What the number cannot carry.** Twenty-seven subjects give a ±18.5 pp interval: 55 and 65 are
the same measurement. This set cannot tell two variants apart, which is why the board is the
five-fold estimate and this is reported once, at the end.

**What to write.** One paragraph, and it should be allowed to be unflattering:

> On the 27-subject test set held out from every fold, `tia` (`late`, 164,737 trainable
> parameters, three seeds × five folds) reaches **ACC xx.x ± y.y, AUC zz.z** (mean over the 15
> models), against the majority-baseline floor of the same clips. The cluster-bootstrap interval
> over subjects is [lo, hi], and the board's five-fold estimate on the working set is
> 56.1 ± 5.9 ACC / 57.3 AUC. The two agree within their intervals, which is all 27 subjects can
> establish.

Fill the placeholders from §7 and §10 — do not retype them by hand; the cell prints them in
the order of the sentence.

**What not to do next.** Do not run this notebook again with another variant to see whether it
does better. The test set is spent; a second number from it is a maximum over draws, and its
interval no longer means what it says.